# Topic Modeling with NMF

Did NMF instead of LDA to get better topics at the suggestion of Prof. Alvarado

## Setup and Imports

In [151]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA, NMF 

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD

In [152]:
sns.set_theme(style="white")
colors = "YlGnBu"

In [153]:
model_type = 'lda' # or 'nmf'
data_home = "../input"


In [154]:
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

In [155]:
OHCO = ['doc_title', 'chunk_id','token_id']
CHUNKS = OHCO[:2]
STORIES = OHCO[:1]

BAG = CHUNKS

In [156]:
LIB = pd.read_csv('data/pg2591-LIB.csv',)
LIB.set_index('doc_title', inplace=True)
LIB.head()

,volume
doc_title,
THE GOLDEN BIRD,1
HANS IN LUCK,1
JORINDA AND JORINDEL,1
THE TRAVELLING MUSICIANS,1
OLD SULTAN,1


In [157]:
TOKENS = pd.read_csv('data/chunked_tokens.csv').set_index(OHCO).dropna()
TOKENS

pos_tuple  pos token_str term_str pos_group
doc_title chunk_id token_id                                                   
ASHPUTTEL 0        0           ('the', 'DT')   DT       the      the        DT
                   1          ('wife', 'NN')   NN      wife     wife        NN
                   2            ('of', 'IN')   IN        of       of        IN
                   3             ('a', 'DT')   DT         a        a        DT
                   4          ('rich', 'JJ')   JJ      rich     rich        JJ
...                                      ...  ...       ...      ...       ...
TOM THUMB 19       66           ('s', 'VBZ')  VBZ         s        s        VB
                   67           ('no', 'DT')   DT        no       no        DT
                   68        ('place', 'NN')   NN     place    place        NN
                   69         ('like', 'IN')   IN      like     like        IN
                   70         ('home', 'NN')   NN      home     home        NN

[115213 rows x 5 columns]

In [158]:
DOCS = TOKENS[TOKENS.pos.str.match(r'^NNS?$')]\
    .groupby(BAG).term_str\
    .apply(lambda x: ' '.join(map(str,x)))\
    .to_frame()\
    .rename(columns={'term_str':'doc_str'})

DOCS

doc_str
doc_title chunk_id                                                   
ASHPUTTEL 0         wife man end drew daughter bedside girl i watc...
          1         fair face foul heart sorry time girl goodforno...
          2         hearth ashes course dirty ashputtel father wif...
          3         daughter mother s grave tears tree times day b...
          4         hair shoes sashes king s feast ball mother not...
...                                                               ...
TOM THUMB 15        wolf chat friend i treat s wolf house father s...
          16        content way tom shout noise wolf everybody hou...
          17        wolf woodman axe wife scythe do woodman i head...
          18        ah father world i way home air father i mouseh...
          19        clothes ones journey home father mother peace ...

[784 rows x 1 columns]

## Create Vector Space

In [159]:
from sklearn.feature_extraction import text

my_stop_words = list(text.ENGLISH_STOP_WORDS.union(['yes']))
my_stop_words[:10]
# my_stop_words.append('said') # add more words to stop words because they appeared it most topics and ruined the topics
# my_stop_words.append('came')
# my_stop_words.append('went')

['whereafter',
 'all',
 'becomes',
 'least',
 'above',
 'what',
 'thence',
 'again',
 'its',
 'beside']

In [160]:
count_engine = CountVectorizer(max_df=.75, min_df=5, stop_words=my_stop_words) # Got some advice from clause to lower min ax max df because corpus ins amll
count_model = count_engine.fit_transform(DOCS.doc_str)
TERMS = count_engine.get_feature_names_out()
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'
DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)
DTM

account  advice  ah  air  alas  ale  anger  animals  \
doc_title chunk_id                                                        
ASHPUTTEL 0               0       0   0    0     0    0      0        0   
          1               0       0   0    0     0    0      0        0   
          2               0       0   0    0     0    0      0        0   
          3               0       0   0    0     0    0      0        0   
          4               0       0   0    0     0    0      0        0   
...                     ...     ...  ..  ...   ...  ...    ...      ...   
TOM THUMB 15              0       0   0    0     0    0      0        0   
          16              0       0   0    0     0    0      0        0   
          17              0       0   1    0     0    0      0        0   
          18              0       0   1    1     0    0      0        0   
          19              0       0   0    0     0    0      0        0   

                    answer  apple  ...  woods  word  words  work  world  \
doc_title chunk_id                 ...                                    
ASHPUTTEL 0              0      0  ...      0     0      0     0      0   
          1              0      0  ...      0     0      0     1      0   
          2              0      0  ...      0     0      0     0      0   
          3              0      0  ...      0     0      0     0      0   
          4              0      0  ...      0     0      0     0      0   
...                    ...    ...  ...    ...   ...    ...   ...    ...   
TOM THUMB 15             0      0  ...      0     0      0     0      0   
          16             0      0  ...      0     0      0     0      0   
          17             0      0  ...      0     0      0     0      0   
          18             0      0  ...      0     0      0     0      2   
          19             0      0  ...      0     0      0     0      0   

                    wretch  yard  year  years  youth  
doc_title chunk_id                                    
ASHPUTTEL 0              0     0     0      0      0  
          1              0     0     0      0      0  
          2              0     0     0      0      0  
          3              0     0     0      0      0  
          4              0     0     0      0      0  
...                    ...   ...   ...    ...    ...  
TOM THUMB 15             0     0     0      0      0  
          16             0     0     0      0      0  
          17             0     0     0      0      0  
          18             0     0     0      0      0  
          19             0     0     0      0      0  

[784 rows x 707 columns]

In [161]:
# Used claude code to help with tfidf engine and model because I want nmf to get better topics than lda
# tfidf_engine = TfidfVectorizer(max_df=.75, min_df=5, stop_words=my_stop_words)
# tfidf_model = tfidf_engine.fit_transform(DOCS.doc_str)
# TERMS = tfidf_engine.get_feature_names_out()
# TFIDF = tfidf_engine.fit_transform(DOCS.doc_str)
# VOCAB = pd.DataFrame(index=TERMS)
# VOCAB.index.name = 'term_str'
# DTM = pd.DataFrame(tfidf_model.toarray(), index=DOCS.index, columns=TERMS)
# DTM


## Generate Model with 20 Topics

In [162]:
n_topics = 5
max_iter = 100
n_top_terms = 5
TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]

In [163]:
if model_type == 'lda':
    topic_engine = LDA(n_components=n_topics, max_iter=max_iter)
elif model_type == 'nmf':
    topic_engine = NMF(n_components=n_topics, max_iter=max_iter)
topic_model = topic_engine.fit_transform(count_model)

In [164]:
model_type

'lda'

## THETA

In [165]:
THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
THETA.columns.name = 'topic_id'
THETA.sample(10).T.style.background_gradient(cmap=colors, axis=None)

In [166]:
THETA_vol=THETA.join(LIB)
THETA_vol.groupby('volume').mean().style.background_gradient(cmap=colors, axis=None)



,T0,T1,T2,T3,T4
volume,,,,,
1,0.199286,0.240972,0.168217,0.205942,0.185583
2,0.212035,0.203288,0.099241,0.080462,0.404974


## PHI

In [167]:
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'
PHI.T.sample(10).T.style.background_gradient(cmap=colors, axis=None)

term_str,valley,beauty,water,stay,window,animals,arm,supper,tears,forwards
topic_id,,,,,,,,,,
T0,1.199547,0.200213,20.954597,2.512913,23.636217,2.177196,2.222776,0.207587,3.538031,5.576461
T1,0.200088,7.037575,9.336095,0.200383,9.273368,0.200319,6.166980,0.200007,11.857208,1.819875
T2,0.200004,0.202020,12.190764,0.200007,6.051029,8.210563,0.200405,0.206134,0.201355,0.200569
T3,0.200004,2.357504,14.676479,0.202173,13.835394,0.211918,0.201203,4.189370,0.201125,0.200005
T4,4.200357,0.202689,74.842065,2.884524,0.203991,0.200005,0.208637,5.196901,0.202281,0.203089


## Get Top Terms By Topic

In [168]:
TOPICS = PHI.stack().groupby('topic_id')\
    .apply(lambda x: ' '.join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str))\
    .to_frame('top_terms')
TOPICS


,top_terms
topic_id,
T0,woman wife door children house
T1,king father man son daughter
T2,fox tailor tree bird wolf
T3,hans home day mother time
T4,king princess water dwarf man


## PCA + LDA

In [169]:
THETA.join(LIB)

T0        T1        T2        T3        T4  volume
doc_title chunk_id                                                          
ASHPUTTEL 0         0.008585  0.746231  0.008478  0.120773  0.115932       1
          1         0.009442  0.009328  0.962523  0.009391  0.009316       1
          2         0.008231  0.453346  0.522114  0.008125  0.008184       1
          3         0.011220  0.954671  0.011569  0.011268  0.011272       1
          4         0.016804  0.524984  0.017035  0.016808  0.424369       1
...                      ...       ...       ...       ...       ...     ...
TOM THUMB 15        0.465152  0.011391  0.332033  0.180104  0.011320       1
          16        0.583340  0.375672  0.013818  0.013697  0.013471       1
          17        0.009724  0.795994  0.174899  0.009777  0.009607       1
          18        0.012752  0.686752  0.274793  0.012790  0.012913       1
          19        0.016966  0.294850  0.017067  0.017322  0.653795       1

[784 rows x 6 columns]

In [170]:
pca_engine = PCA(n_components=3)
DCM_THETA = pd.DataFrame(pca_engine.fit_transform(THETA), index=THETA.index)
DCM_THETA.columns = ['PC{}'.format(i) for i in DCM_THETA.columns]
DCM_THETA = DCM_THETA.join(LIB)
DCM_THETA

PC0       PC1       PC2  volume
doc_title chunk_id                                      
ASHPUTTEL 0        -0.365194 -0.385780 -0.175135       1
          1        -0.108930  0.010005  0.185932       1
          2        -0.316509 -0.217436 -0.058921       1
          3        -0.547082 -0.471513 -0.335951       1
          4         0.023787 -0.369076 -0.204195       1
...                      ...       ...       ...     ...
TOM THUMB 15       -0.095984  0.392694  0.041460       1
          16       -0.243169  0.282016 -0.334017       1
          17       -0.474711 -0.391388 -0.248106       1
          18       -0.420986 -0.333833 -0.188405       1
          19        0.336920 -0.319169 -0.135743       1

[784 rows x 4 columns]

In [171]:
# DCM_THETA['doc_weight_mean']=DCM_THETA.mean(axis=1)
# DCM_THETA

In [172]:
DCM_THETA

PC0       PC1       PC2  volume
doc_title chunk_id                                      
ASHPUTTEL 0        -0.365194 -0.385780 -0.175135       1
          1        -0.108930  0.010005  0.185932       1
          2        -0.316509 -0.217436 -0.058921       1
          3        -0.547082 -0.471513 -0.335951       1
          4         0.023787 -0.369076 -0.204195       1
...                      ...       ...       ...     ...
TOM THUMB 15       -0.095984  0.392694  0.041460       1
          16       -0.243169  0.282016 -0.334017       1
          17       -0.474711 -0.391388 -0.248106       1
          18       -0.420986 -0.333833 -0.188405       1
          19        0.336920 -0.319169 -0.135743       1

[784 rows x 4 columns]

In [173]:
def vis_pcs(a, b,DCM, col='doc_label'):
    fig =px.scatter(DCM, 
        f"PC{a}", f"PC{b}", 
        color=DCM[col].astype('category'), 
        size= DCM['doc_weight_mean'],
        hover_name=DCM.title_para,
        marginal_x='box', 
        height=1000, 
        width=1200)
    return fig


In [174]:
DCM_THETA['title']=DCM_THETA.index.get_level_values(0)
DCM_THETA['para']=DCM_THETA.index.get_level_values(1) # Used AI to help extract index values
DCM_THETA['title_para']=DCM_THETA['title']+' '+DCM_THETA['para'].astype(str)
DCM_THETA

PC0       PC1       PC2  volume      title  para  \
doc_title chunk_id                                                          
ASHPUTTEL 0        -0.365194 -0.385780 -0.175135       1  ASHPUTTEL     0   
          1        -0.108930  0.010005  0.185932       1  ASHPUTTEL     1   
          2        -0.316509 -0.217436 -0.058921       1  ASHPUTTEL     2   
          3        -0.547082 -0.471513 -0.335951       1  ASHPUTTEL     3   
          4         0.023787 -0.369076 -0.204195       1  ASHPUTTEL     4   
...                      ...       ...       ...     ...        ...   ...   
TOM THUMB 15       -0.095984  0.392694  0.041460       1  TOM THUMB    15   
          16       -0.243169  0.282016 -0.334017       1  TOM THUMB    16   
          17       -0.474711 -0.391388 -0.248106       1  TOM THUMB    17   
          18       -0.420986 -0.333833 -0.188405       1  TOM THUMB    18   
          19        0.336920 -0.319169 -0.135743       1  TOM THUMB    19   

                      title_para  
doc_title chunk_id                
ASHPUTTEL 0          ASHPUTTEL 0  
          1          ASHPUTTEL 1  
          2          ASHPUTTEL 2  
          3          ASHPUTTEL 3  
          4          ASHPUTTEL 4  
...                          ...  
TOM THUMB 15        TOM THUMB 15  
          16        TOM THUMB 16  
          17        TOM THUMB 17  
          18        TOM THUMB 18  
          19        TOM THUMB 19  

[784 rows x 7 columns]

## Save Files to Output

In [175]:
THETA.to_csv(f"{output_dir}/pg2591-THETA.csv", index=True)
PHI.to_csv(f"{output_dir}/pg2591-PHI.csv", index=True)
TOPICS.to_csv(f"{output_dir}/pg2591-TOPICS.csv", index=True)